Multi-head self-attention mixes information across tokens: each output is a weighted average of value vectors. The weights depend non-linearly on the input, through the softmax, but each output is still a weighted average of linear projections of the tokens. The MLP adds a separate, feature-wise non-linear transformation that is applied to each token on its own, after the tokens have exchanged information. In a conventional dense transformer with hidden width $4d$, the MLP has $8d^2$ weights against $4d^2$ for the attention projections, about two thirds of the matrix weights of the block, not counting biases and normalization parameters.

Continuing the notation of the [multi-head attention](/aiml-common/lectures/nlp/transformers/multihead-self-attention/index) section, where $\hat Z$ is the output of the attention block, the MLP sub-block is

$$
\begin{aligned}
Y &= \mathtt{LayerNorm}(\hat Z) \\
\tilde X &= \mathtt{MLP}(Y) + \hat Z
\end{aligned}
$$

As in the attention block, a skip connection adds the input to the MLP output. The MLP itself is two linear layers with an activation in between, usually a ReLU or a GELU. The GELU weighs its input by the probability that a standard Gaussian variable is smaller, $\mathrm{GELU}(x) = x\,\Phi(x)$, so it is a smooth gating function resembling ReLU but not monotone everywhere.

In [ ]:
import matplotlib.pyplot as plt
import torch

x = torch.linspace(-4, 4, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(x, torch.relu(x), label="ReLU")
ax.plot(x, torch.nn.functional.gelu(x), label="GELU")
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("x")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Unlike the ReLU, the GELU is smooth at 0 and slightly negative for small negative inputs.

The code below implements the MLP, also called the feed-forward layer, with the feed-forward sizes of BERT-base (which uses post-LN, unlike the pre-LN equations above): $d = 768$ and a hidden width of $4d = 3072$.

In [ ]:
from types import SimpleNamespace

import torch.nn as nn

torch.manual_seed(0)


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.linear_1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.linear_2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, x):
        x = self.linear_1(x)
        x = self.gelu(x)
        x = self.linear_2(x)
        x = self.dropout(x)
        return x


config = SimpleNamespace(hidden_size=768, intermediate_size=3072, hidden_dropout_prob=0.1)
ff = FeedForward(config).eval()  # eval mode turns dropout off

d = config.hidden_size
mlp_weights = ff.linear_1.weight.numel() + ff.linear_2.weight.numel()
attention_weights = 4 * d * d  # W^q, W^k, W^v and W^o
print(f"MLP weights:       {mlp_weights:,} = {mlp_weights / d**2:.0f} d^2")
print(f"attention weights: {attention_weights:,} = 4 d^2")
print(f"MLP share of the block: {mlp_weights / (mlp_weights + attention_weights):.2f}")

In evaluation mode (dropout disabled), the MLP processes each token independently of the other tokens. If the tokens of the input are shuffled, its outputs are shuffled in the same way and are otherwise unchanged. The next cell checks this.

In [ ]:
x = torch.randn(10, d)  # 10 tokens
perm = torch.randperm(10)
with torch.no_grad():
    same = torch.allclose(ff(x)[perm], ff(x[perm]), atol=1e-6)
print("shuffling the tokens only shuffles the outputs:", same)

In this standard block, attention is the only operation that directly exchanges information across token positions. The MLP then transforms each token's embedding with the same weights.

## Resources

1. [Transformers Explained Visually, Part 2: How it works, step by step](https://towardsdatascience.com/transformers-explained-visually-part-2-how-it-works-step-by-step-b49fa4a64f34)

2. [Transformers Explained Visually, Part 3: Multi-head attention deep dive](https://towardsdatascience.com/transformers-explained-visually-part-3-multi-head-attention-deep-dive-1c1ff1024853)

## PyTorch reference

| PyTorch class | Description |
|---|---|
| [`nn.Linear`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Linear.html) | Affine map of the input: $y = xA^T + b$. |
| [`nn.GELU`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.GELU.html) | Gaussian Error Linear Unit activation. |
| [`nn.ReLU`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.ReLU.html) | Rectified linear unit, applied element-wise. |
| [`nn.Dropout`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Dropout.html) | During training, zeroes random elements of the input with probability $p$. |
| [`nn.LayerNorm`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.LayerNorm.html) | Layer normalization over the last dimensions of the input. |

Key references: [@Vaswani2017-vd; @Ramachandran2017-yd]; Hendrycks and Gimpel, Gaussian Error Linear Units (GELUs), 2016, [arXiv:1606.08415](https://arxiv.org/abs/1606.08415).